In [1]:
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
from skimage.draw import polygon
import matplotlib.pyplot as plt
import sys


In [2]:
path_dice_ensemble = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ensemble/segmentation_results_ensemble_dice_adults.csv'
path_dice_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg/segmentation_results_phiseg_dice_adults.csv'
path_dice_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids/segmentation_results_vids_dice_adults_7.csv'

df_ens = pd.read_csv(path_dice_ensemble)
df_phiseg = pd.read_csv(path_dice_phiseg)
df_vids = pd.read_csv(path_dice_vids)

In [3]:
df_vids.head()

,filename,age,phase,dice
0,CR32a7581-CR32a9b0b-000051.h5,18,ed,0.903226
1,CR32a7581-CR32a9b0b-000051.h5,18,es,0.905998
2,CR32a9679-CR3dcb61c-000048.h5,18,ed,0.784466
3,CR32a9679-CR3dcb61c-000048.h5,18,es,0.919181
4,CR3dcac3c-CR3dcaf03-000043.h5,15,ed,0.942286


In [4]:
# Define age group bins and labels
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

# Create age group column
df_ens['age_group'] = pd.cut(df_ens['age'], bins=bins, labels=labels, right=True)

# Compute mean and std of dice per age group
stats = df_ens.groupby('age_group', observed=False)['dice'].agg(['mean', 'std', 'count'])
stats.columns = ['mean_dice', 'std_dice', 'n_samples']

print(stats)

             mean_dice  std_dice  n_samples
age_group                                  
infant        0.781360  0.264063         24
toddler       0.897486  0.054473         58
preschooler   0.898653  0.047725         84
school age    0.891283  0.066490        200
teenager      0.883828  0.074854        262


In [5]:
# Define age group bins and labels
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

# Create age group column
df_phiseg['age_group'] = pd.cut(df_phiseg['age'], bins=bins, labels=labels, right=True)

# Compute mean and std of dice per age group
stats = df_phiseg.groupby('age_group', observed=False)['dice'].agg(['mean', 'std', 'count'])
stats.columns = ['mean_dice', 'std_dice', 'n_samples']

print(stats)

             mean_dice  std_dice  n_samples
age_group                                  
infant        0.751998  0.270459         24
toddler       0.865494  0.132888         58
preschooler   0.884746  0.068174         84
school age    0.879901  0.092649        200
teenager      0.878344  0.079164        262


In [6]:
# Define age group bins and labels
bins = [-np.inf, 0, 2, 5, 12, 18]
labels = ['infant', 'toddler', 'preschooler', 'school age', 'teenager']

# Create age group column
df_vids['age_group'] = pd.cut(df_vids['age'], bins=bins, labels=labels, right=True)

# Compute mean and std of dice per age group
stats = df_vids.groupby('age_group', observed=False)['dice'].agg(['mean', 'std', 'count'])
stats.columns = ['mean_dice', 'std_dice', 'n_samples']

print(stats)

             mean_dice  std_dice  n_samples
age_group                                  
infant        0.763025  0.260195         24
toddler       0.875686  0.062382         58
preschooler   0.888090  0.057195         84
school age    0.879774  0.058350        200
teenager      0.878142  0.074260        262


In [7]:
# now for ejection fraction
path_ef_ens = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/ensemble/results_ensemble_ef_adults.csv'
path_ef_phiseg = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/phiseg/results_phiseg_ef_adults.csv'
path_ef_vids = '/home/paul/workspace/year1/projects/ped_unc/echonet/exp/results_data/vids/results_vids_ef_adults_7.csv'

df_ef_ens = pd.read_csv(path_ef_ens)
df_ef_phiseg = pd.read_csv(path_ef_phiseg)
df_ef_vids = pd.read_csv(path_ef_vids)

In [8]:
df_ef_ens.head()

,filename,age,mean_ef_pred,std_ef_pred,gt_ef
0,CR32a7581-CR32a9b0b-000051.h5,18,36.698528,2.017306,44.31
1,CR32a9679-CR3dcb61c-000048.h5,18,34.716423,10.693580,68.16
2,CR3dcac3c-CR3dcaf03-000043.h5,15,63.384238,3.572754,60.61
3,CR32a95c7-CR32a97ae-000046.h5,2,63.783924,18.181288,55.41
4,CR3dcb191-CR3dcb49d-000047.h5,17,32.414221,16.784096,67.31


In [9]:
# Create age group column
df_ef_ens['age_group'] = pd.cut(df_ef_ens['age'], bins=bins, labels=labels, right=True)

# Compute error metrics per sample
df_ef_ens['error'] = df_ef_ens['mean_ef_pred'] - df_ef_ens['gt_ef']  # signed error
df_ef_ens['abs_error'] = np.abs(df_ef_ens['error'])  # absolute error
df_ef_ens['squared_error'] = df_ef_ens['error'] ** 2  # squared error

# Compute stats per age group
stats = df_ef_ens.groupby('age_group', observed=False).agg(
    mae=('abs_error', 'mean'),             # Mean Absolute Error
    mae_std=('abs_error', 'std'),          # Std of Absolute Error
    correlation=('error', lambda x: df_ef_ens.loc[x.index, 'mean_ef_pred'].corr(
                                    df_ef_ens.loc[x.index, 'gt_ef'])),
    n_samples=('filename', 'count')
)

print(stats.round(3))

                mae  mae_std  correlation  n_samples
age_group                                           
infant       17.574   27.817        0.365         12
toddler      11.376   12.882        0.648         29
preschooler   8.155    8.850        0.613         42
school age   13.486   28.437        0.110        100
teenager     13.937   15.882        0.556        131


In [10]:
# Create age group column
df_ef_phiseg['age_group'] = pd.cut(df_ef_phiseg['age'], bins=bins, labels=labels, right=True)

# Compute error metrics per sample
df_ef_phiseg['error'] = df_ef_phiseg['mean_ef_pred'] - df_ef_phiseg['gt_ef']  # signed error
df_ef_phiseg['abs_error'] = np.abs(df_ef_phiseg['error'])  # absolute error
df_ef_phiseg['squared_error'] = df_ef_phiseg['error'] ** 2  # squared error

# Compute stats per age group
stats_phiseg = df_ef_phiseg.groupby('age_group', observed=False).agg(
    mae=('abs_error', 'mean'),             # Mean Absolute Error
    mae_std=('abs_error', 'std'),          # Std of Absolute Error
    correlation=('error', lambda x: df_ef_phiseg.loc[x.index, 'mean_ef_pred'].corr(
                                    df_ef_phiseg.loc[x.index, 'gt_ef'])),
    n_samples=('filename', 'count')
)

print(stats_phiseg.round(3))

                mae  mae_std  correlation  n_samples
age_group                                           
infant       21.440   29.570        0.375         12
toddler      15.292   15.526        0.441         29
preschooler  10.342   10.960        0.504         42
school age   11.117   12.121        0.287        100
teenager     16.102   36.307        0.260        131


In [11]:
# Create age group column
df_ef_vids['age_group'] = pd.cut(df_ef_vids['age'], bins=bins, labels=labels, right=True)

# Compute error metrics per sample
df_ef_vids['error'] = df_ef_vids['mean_ef_pred'] - df_ef_vids['gt_ef']  # signed error
df_ef_vids['abs_error'] = np.abs(df_ef_vids['error'])  # absolute error
df_ef_vids['squared_error'] = df_ef_vids['error'] ** 2  # squared error

# Compute stats per age group
stats_vids = df_ef_vids.groupby('age_group', observed=False).agg(
    mae=('abs_error', 'mean'),             # Mean Absolute Error
    mae_std=('abs_error', 'std'),          # Std of Absolute Error
    correlation=('error', lambda x: df_ef_vids.loc[x.index, 'mean_ef_pred'].corr(
                                    df_ef_vids.loc[x.index, 'gt_ef'])),
    n_samples=('filename', 'count')
)

print(stats_vids.round(3))

                  mae   mae_std  correlation  n_samples
age_group                                              
infant       1635.768  1554.720       -0.477         12
toddler       860.813   917.588       -0.168         29
preschooler   995.113  1114.909        0.053         42
school age    983.735  1379.817       -0.061        100
teenager      852.386   805.733        0.049        131
